# cell-curator quickstart

Annotate pre-clustered data from a notebook, end to end, on a synthetic dataset
built in this notebook — nothing to download.

The point of the package is that **no automated call is a verdict**. Every step
below produces evidence; deciding what it means stays with you, or with the
Claude/Codex agent driving the skill. There is deliberately no
`annotate_clusters()` that hands back finished labels, and nothing here contacts
an inference service.

Two habits this notebook models, because both bite people:

1. `output_root` is an **absolute** path. Relative paths are anchored to the
   configuration file's directory, so they work — but being explicit means a
   `chdir` or a kernel restart can never surprise you.
2. Every gate is **declared up front** (`reviewer`, `assumptions`, and the full
   biological context). They are not skipped; an unattended run just has to state
   its answers in advance.

In [ ]:
import logging
import tempfile
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd

from cell_curator import CellCurator

# Phase-boundary progress. The library is silent until you ask for this.
logging.getLogger("cell_curator").setLevel(logging.INFO)
logging.getLogger("cell_curator").addHandler(logging.StreamHandler())

workspace = Path(tempfile.mkdtemp(prefix="cell-curator-quickstart-"))
print("workspace:", workspace)

## 1. A synthetic pre-clustered dataset

Two clean populations, six genes, a fixed seed. `cell-curator` annotates
*existing* clusters — it never clusters for you — so `leiden` here stands in for
whatever partition you already trust.

In [ ]:
rng = np.random.default_rng(7)
n_per_cluster = 40
total = n_per_cluster * 2

counts = np.zeros((total, 6), dtype=np.float32)
counts[:n_per_cluster, :2] = rng.poisson(9, size=(n_per_cluster, 2)) + 1
counts[n_per_cluster:, 2:4] = rng.poisson(9, size=(n_per_cluster, 2)) + 1
genes = ["CD3D", "CD3E", "MS4A1", "CD79A", "NOISE1", "NOISE2"]

adata = ad.AnnData(
    X=np.log1p(counts),
    obs=pd.DataFrame(
        {"leiden": ["0"] * n_per_cluster + ["1"] * n_per_cluster},
        index=[f"cell-{index:03d}" for index in range(total)],
    ),
    var=pd.DataFrame({"gene_symbol": genes}, index=genes),
)
adata.layers["counts"] = counts
adata.layers["logcounts"] = np.log1p(counts)
adata.obsm["X_pca_harmony"] = np.vstack(
    [
        rng.normal(-3, 0.2, size=(n_per_cluster, 3)),
        rng.normal(3, 0.2, size=(n_per_cluster, 3)),
    ]
).astype(np.float32)

adata

## 2. Signed marker programs

Each label carries positives **and** negatives. The negatives are what let the
package say "this is not a B cell" rather than only "this looks a bit like one".

In [ ]:
markers = {
    "T cell": {"pos": ["CD3D", "CD3E"], "neg": ["MS4A1", "CD79A"]},
    "B cell": {"pos": ["MS4A1", "CD79A"], "neg": ["CD3D", "CD3E"]},
}

## 3. Configure the run

The constructor writes a strict `config.yaml` into the run directory and
validates it immediately, so a mistake surfaces here rather than three phases
later. Your in-memory `adata` is written to that directory and hashed — the
object you hold is never modified.

In [ ]:
curator = CellCurator(
    adata=adata,
    organism="human",
    tissue="blood",
    cluster_key="leiden",
    markers=markers,
    run_id="quickstart-v1",
    output_root=workspace / "results",
    mode="annotation",
    # Gates, declared rather than skipped:
    preauthorized=True,
    reviewer="quickstart-reviewer",
    assumptions=["The synthetic marker programs define the L1 vocabulary."],
    # Context the package refuses to guess:
    developmental_stage="adult",
    condition="healthy",
    experimental_context="synthetic quickstart fixture",
)
curator

In [ ]:
# Read-only: reports what was found and changes nothing.
inspection = curator.inspect()
{key: inspection[key] for key in sorted(inspection) if key != "obsm"}

## 4. Freeze the input, then set the scene

`prepare()` hash-freezes the input, records the biological context and detected
compute, and builds the plan. Evidence generation requires both stages, so this
is the one ordering you do not have to remember.

In [ ]:
prepared = curator.prepare()
curator.status()

## 5. Look for impure clusters before splitting anything

`audit_parents()` reports the signals that would justify subclustering — donor
and capture dominance, doublet fraction, incompatible-program fraction, technical
outliers. A flag is a reason to look closer, never a decision.

In [ ]:
audit = curator.audit_parents()
audit[[column for column in ("scope", "parent_id", "n_cells") if column in audit]]

## 6. Evidence

Per-cluster program evidence and bottom-up markers. This is the output you (or
the agent) actually reason over.

In [ ]:
evidence, bottom_up = curator.evidence()
evidence.head()

## 7. Finish the run

`annotate()` runs every remaining computational phase and freezes an immutable
review packet. It is idempotent, so it resumes the work already done above
rather than repeating it.

Note what it does *not* do: it writes no labels into your object. That needs
`write_back()`, and `write_back()` needs an explicit approval file.

In [ ]:
packet = curator.annotate()
print("review packet:", packet)
sorted(item.name for item in packet.iterdir())

In [ ]:
curator.validate_run()

## 8. Where to go next

- `curator.propose_subclusters()` / `decide_subclusters()` — adaptive refinement,
  where a parent splits only if the evidence gates pass. Impurity that turns out
  to be technical or doublet-driven is reported as such, not carved into subtypes.
- `curator.build_critic_packet()` — the adversarial review surface.
- `curator.preview_writeback(...)` then `curator.write_back(...)` — guarded,
  approval-gated write-back to a new object.
- Every CLI capability has a Python equivalent; see `cell_curator.__all__`.

In [ ]:
import cell_curator

print(len(cell_curator.__all__), "public names")
[name for name in cell_curator.__all__ if "crosscheck" in name or "map_" in name]